# Laptop

In [ ]:
# 1 — Imports & Setup
import os
import re
import json
import time
import random
import requests
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple, Optional
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizerFast, RobertaModel
from torch.optim import AdamW
from sklearn.model_selection import KFold
from scipy.stats import pearsonr
from tqdm import tqdm

# 2 — Config
class Config:
    # Task
    SUBTASK = "subtask_2"
    LANG = "eng"
    DOMAINS = ["laptop"]

    # Model
    MODEL_NAME = "roberta-base"
    MAX_LEN_EXTRACTION = 512
    MAX_LEN_PAIR = 512

    # Train
    BATCH_SIZE_EXTRACTION = 8
    BATCH_SIZE_PAIR = 16
    MAX_EPOCHS_EXTRACTION = 3
    MAX_EPOCHS_PAIR = 3
    LR = 2e-5
    SEED = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # OOF negatives
    OOF_K = 5
    OOF_EPOCHS_EXTRACTION = 3
    MAX_WRONG_SPANS_PER_SENT = 3
    MAX_NEG_PAIRS_PER_SENT = 10

    # NULL handling
    USE_NULL_TOKEN = True
    NULL_TOKEN = "[NULL]"

    # add hard unpaired negatives from wrong spans
    ADD_UNPAIRED_FROM_WRONG = True
    MAX_UNPAIRED_PER_SENT = 8

    # Outer fold
    OUTER_K = 5

    @classmethod
    def get_train_url(cls, domain: str) -> str:
        filename = f"{cls.LANG}_{domain}_train_alltasks.jsonl"
        return (
            "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/"
            f"task-dataset/track_a/{cls.SUBTASK}/{cls.LANG}/{filename}"
        )

    @classmethod
    def get_local_filename(cls, domain: str) -> str:
        return f"{cls.LANG}_{domain}_train_alltasks.jsonl"

    @classmethod
    def output_metrics_csv(cls, domain: str) -> str:
        return f"dimaste_kfold_metrics_{domain}.csv"

    @classmethod
    def output_predictions_csv(cls, domain: str) -> str:
        return f"dimaste_predictions_output_{domain}.csv"

config = Config()

TAG2IDX = {'O': 0, 'B-ASP': 1, 'I-ASP': 2, 'B-OP': 3, 'I-OP': 4}
IDX2TAG = {v: k for k, v in TAG2IDX.items()}

D_MAX = np.sqrt(8**2 + 8**2)
NULL_TOKEN = config.NULL_TOKEN

print(f"Running on device: {config.DEVICE}")
print(f"Domains: {config.DOMAINS}")
print(f"NULL token enabled: {config.USE_NULL_TOKEN}")
print(f"Epochs: extractor={config.MAX_EPOCHS_EXTRACTION}, pair={config.MAX_EPOCHS_PAIR}, oof_extractor={config.OOF_EPOCHS_EXTRACTION}")
print(f"Folds: outer={config.OUTER_K}, inner_oof={config.OOF_K}")

# 3 — Seed & Helpers
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.SEED)

def format_duration(seconds: float) -> str:
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

def safe_text(x: Any) -> str:
    return x if isinstance(x, str) else ""

def norm_text(s: Any) -> str:
    s = safe_text(s)
    return re.sub(r"\s+", " ", s.strip().lower())

def is_null(x: Any) -> bool:
    if x is None:
        return True
    if isinstance(x, str):
        t = x.strip()
        if t.upper() == "NULL":
            return True
        if config.USE_NULL_TOKEN and t == NULL_TOKEN:
            return True
    return False

def normalize_candidate_token(x: Any) -> str:
    # Map internal [NULL] token to dataset-style "NULL"
    if isinstance(x, str) and x.strip() == NULL_TOKEN:
        return "NULL"
    return safe_text(x)

def dedup_triplets_exact(triplets: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for t in triplets:
        a = t.get("aspect")
        o = t.get("opinion")
        v = t.get("valence")
        ar = t.get("arousal")
        key = (
            norm_text(a) if isinstance(a, str) else str(a),
            norm_text(o) if isinstance(o, str) else str(o),
            round(float(v), 6) if v is not None else None,
            round(float(ar), 6) if ar is not None else None,
        )
        if key in seen:
            continue
        seen.add(key)
        out.append(t)
    return out

def calculate_normalized_euclidean_distance(vp, ap, vg, ag) -> float:
    dist = np.sqrt((vp - vg)**2 + (ap - ag)**2) / D_MAX
    return float(np.clip(dist, 0.0, 1.0))

def find_all_spans(text: str, phrase: str) -> List[Tuple[int, int]]:
    if not phrase:
        return []
    return [(m.start(), m.end()) for m in re.finditer(re.escape(phrase), text)]

def pick_closest_pair(
    asp_spans: List[Tuple[int, int]],
    op_spans: List[Tuple[int, int]]
) -> Optional[Tuple[int, int, int, int]]:
    best = None
    best_dist = 10**18
    for a_s, a_e in asp_spans:
        a_c = (a_s + a_e) / 2
        for o_s, o_e in op_spans:
            o_c = (o_s + o_e) / 2
            d = abs(a_c - o_c)
            if d < best_dist:
                best_dist = d
                best = (a_s, a_e, o_s, o_e)
    return best

def roberta_tokens_to_text(tokens: List[str]) -> str:
    out = ""
    for t in tokens:
        if t in ["<s>", "</s>", "<pad>"]:
            continue
        if t.startswith("Ġ"):
            out += (" " + t[1:]) if out else t[1:]
        else:
            out += t
    return out.strip()

def uniq_keep_order(xs: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in xs:
        k = norm_text(x)
        if not k:
            continue
        if k in seen:
            continue
        seen.add(k)
        out.append(x)
    return out

# 4 — Download & Load
def download_and_load(url: str, filename: str) -> List[Dict[str, Any]]:
    if not os.path.exists(filename):
        last_err = None
        for attempt in range(3):
            try:
                r = requests.get(url, timeout=60)
                r.raise_for_status()
                with open(filename, "wb") as f:
                    f.write(r.content)
                break
            except Exception as e:
                last_err = e
                print(f"  download attempt {attempt+1}/3 failed: {e}")
                time.sleep(2 + attempt)
        else:
            raise RuntimeError(f"Failed downloading after retries: {last_err}")

    data = []
    total_triplets_initial = 0
    total_triplets_deduped = 0

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            entry = json.loads(line)

            triplets = []
            for quad in entry.get("Quadruplet", []):
                asp = quad.get("Aspect", "NULL")
                op = quad.get("Opinion", "NULL")

                v = a = None
                va_raw = quad.get("VA", None)
                if isinstance(va_raw, str) and "#" in va_raw:
                    try:
                        v, a = map(float, va_raw.split("#"))
                    except Exception:
                        v, a = None, None

                triplets.append({
                    "aspect": asp,
                    "opinion": op,
                    "valence": v,
                    "arousal": a,
                })

            total_triplets_initial += len(triplets)
            triplets = dedup_triplets_exact(triplets)
            total_triplets_deduped += len(triplets)

            data.append({
                "id": entry.get("ID"),
                "text": entry.get("Text", ""),
                "triplets": triplets,
            })

    print("\nLoad Summary")
    print(f"Items                 : {len(data)}")
    print(f"Triplets (initial)    : {total_triplets_initial}")
    print(f"Triplets (deduped)    : {total_triplets_deduped}")
    return data

# 5 — Cleaning + Stats
def cleaning_with_stats(data: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    original = len(data)

    texts = [norm_text(x.get("text", "")) for x in data]
    counts = {}
    for t in texts:
        counts[t] = counts.get(t, 0) + 1
    dup_total = sum(c - 1 for c in counts.values() if c > 1)

    # Remove duplicates (keep first)
    seen = set()
    deduped = []
    for it in data:
        t = norm_text(it.get("text", ""))
        if t in seen:
            continue
        seen.add(t)
        deduped.append(it)
    after_dedup = len(deduped)

    # Missing/empty definition: invalid text OR empty triplets list
    missing_or_empty = 0
    cleaned = []
    for it in deduped:
        t = safe_text(it.get("text", ""))
        invalid_text = (
            (t is None) or
            (t == "") or
            (t.strip() == "") or
            (t.strip().upper() in {"NULL", "NONE"}) or
            (t.strip().lower() == "nan")
        )
        empty_triplets = not isinstance(it.get("triplets", []), list) or len(it.get("triplets", [])) == 0

        if invalid_text or empty_triplets:
            missing_or_empty += 1
            continue
        cleaned.append(it)

    after_remove_missing = len(cleaned)

    print("\nDATA CLEANING STATS")
    print(f"Jumlah data asli                                 : {original}")
    print(f"Jumlah data duplikat (berdasarkan Text)           : {dup_total}")
    print(f"Jumlah data setelah hapus duplikat                : {after_dedup}")
    print(f"Jumlah data missing/kosong (Text invalid/Triplet empty) : {missing_or_empty}")
    print(f"Jumlah data setelah hapus missing/kosong          : {after_remove_missing}")

    stats = {
        "original": original,
        "duplicates": dup_total,
        "after_dedup": after_dedup,
        "missing_or_empty": missing_or_empty,
        "after_remove_missing": after_remove_missing,
    }
    return cleaned, stats

def filter_trainable_items(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Trainable = punya minimal 1 triplet dengan VA valid.
    Keep paired & unpaired (NULL) as long as VA exists.
    """
    out = []
    for it in items:
        good = []
        for t in it.get("triplets", []):
            v = t.get("valence")
            a = t.get("arousal")
            if v is None or a is None:
                continue
            good.append({
                "aspect": normalize_candidate_token(t.get("aspect")),
                "opinion": normalize_candidate_token(t.get("opinion")),
                "valence": float(v),
                "arousal": float(a),
            })
        if good:
            out.append({"id": it["id"], "text": it["text"], "triplets": good})
    return out

# 6 — Extraction Dataset/Model (BIO)
class ExtractionDataset(Dataset):
    def __init__(self, data: List[Dict[str, Any]], tokenizer: RobertaTokenizerFast, max_len: int):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.special_ids = {tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = self.data[idx]
        text = safe_text(item.get("text", ""))

        if config.USE_NULL_TOKEN:
            aug_text = f"{NULL_TOKEN} {text}"
            prefix_len = len(NULL_TOKEN) + 1
        else:
            aug_text = text
            prefix_len = 0

        enc = self.tokenizer(
            aug_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        input_ids = enc["input_ids"].squeeze(0)
        mask = enc["attention_mask"].squeeze(0)
        offsets = enc["offset_mapping"].squeeze(0).tolist()

        labels = torch.zeros(self.max_len, dtype=torch.long)
        for i in range(self.max_len):
            if int(input_ids[i].item()) in self.special_ids:
                labels[i] = -100

        def overlap(start, end, s, e):
            return (start < e) and (end > s) and (start < end)

        # Label only non-NULL spans (NULL is not a span in text)
        for t in item.get("triplets", []):
            asp = t.get("aspect")
            op = t.get("opinion")
            if is_null(asp) or is_null(op):
                continue

            asp = safe_text(asp)
            op = safe_text(op)

            asp_spans = find_all_spans(text, asp)
            op_spans = find_all_spans(text, op)
            if not asp_spans or not op_spans:
                continue

            best = pick_closest_pair(asp_spans, op_spans)
            if best is None:
                continue
            asp_s, asp_e, op_s, op_e = best

            asp_s += prefix_len
            asp_e += prefix_len
            op_s += prefix_len
            op_e += prefix_len

            asp_first, op_first = True, True
            for i, (st, en) in enumerate(offsets):
                if i >= self.max_len:
                    break
                if int(input_ids[i].item()) in self.special_ids:
                    continue

                if overlap(st, en, asp_s, asp_e):
                    labels[i] = TAG2IDX["B-ASP"] if asp_first else TAG2IDX["I-ASP"]
                    asp_first = False

                if overlap(st, en, op_s, op_e):
                    labels[i] = TAG2IDX["B-OP"] if op_first else TAG2IDX["I-OP"]
                    op_first = False

        return {
            "ids": input_ids,
            "mask": mask,
            "labels": labels,
            "orig_data": json.dumps(item),
        }

class ExtractionModel(nn.Module):
    def __init__(self, n_tags: int, tokenizer: RobertaTokenizerFast):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(config.MODEL_NAME)
        self.encoder.resize_token_embeddings(len(tokenizer))
        self.drop = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, n_tags)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        seq = self.drop(out.last_hidden_state)
        return self.classifier(seq)

def loss_extraction(logits, labels) -> torch.Tensor:
    return nn.CrossEntropyLoss(ignore_index=-100)(
        logits.view(-1, len(TAG2IDX)),
        labels.view(-1)
    )

def decode_spans_from_tags(
    input_ids: np.ndarray,
    tag_preds: np.ndarray,
    tokenizer: RobertaTokenizerFast
) -> Tuple[List[str], List[str]]:
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    SPECIAL = {tokenizer.bos_token, tokenizer.eos_token, tokenizer.pad_token}

    aspects, opinions = [], []
    current = None

    for tok, tidx in zip(tokens, tag_preds):
        if tok in SPECIAL:
            if current:
                (aspects if current["type"] == "ASP" else opinions).append(current)
                current = None
            continue

        tag = IDX2TAG.get(int(tidx), "O")
        if tag.startswith("B-"):
            if current:
                (aspects if current["type"] == "ASP" else opinions).append(current)
            current_type = "ASP" if "ASP" in tag else "OP"
            current = {"type": current_type, "raw_tokens": [tok]}
        elif tag.startswith("I-") and current:
            if ("ASP" in tag and current["type"] == "ASP") or ("OP" in tag and current["type"] == "OP"):
                current["raw_tokens"].append(tok)
            else:
                (aspects if current["type"] == "ASP" else opinions).append(current)
                current = None
        else:
            if current:
                (aspects if current["type"] == "ASP" else opinions).append(current)
                current = None

    if current:
        (aspects if current["type"] == "ASP" else opinions).append(current)

    asp_txt = [roberta_tokens_to_text(x["raw_tokens"]) for x in aspects]
    op_txt = [roberta_tokens_to_text(x["raw_tokens"]) for x in opinions]

    asp_txt = uniq_keep_order([normalize_candidate_token(x) for x in asp_txt if x])
    op_txt = uniq_keep_order([normalize_candidate_token(x) for x in op_txt if x])
    return asp_txt, op_txt

def train_extractor(train_data, tokenizer, epochs: int) -> ExtractionModel:
    ds = ExtractionDataset(train_data, tokenizer, config.MAX_LEN_EXTRACTION)
    dl = DataLoader(ds, batch_size=config.BATCH_SIZE_EXTRACTION, shuffle=True)

    model = ExtractionModel(len(TAG2IDX), tokenizer).to(config.DEVICE)
    opt = AdamW(model.parameters(), lr=config.LR, weight_decay=0.01)

    for epoch in range(epochs):
        model.train()
        tot = 0.0
        for batch in tqdm(dl, desc=f"Extractor Epoch {epoch+1}/{epochs}"):
            ids = batch["ids"].to(config.DEVICE)
            mask = batch["mask"].to(config.DEVICE)
            labels = batch["labels"].to(config.DEVICE)

            opt.zero_grad()
            logits = model(ids, mask)
            loss = loss_extraction(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += float(loss.item())

        print(f"   Epoch {epoch+1} | AvgLoss {tot/max(1,len(dl)):.4f}")
    return model

def extract_candidates(
    model: ExtractionModel,
    data: List[Dict[str, Any]],
    tokenizer: RobertaTokenizerFast
) -> Dict[str, Dict[str, List[str]]]:
    ds = ExtractionDataset(data, tokenizer, config.MAX_LEN_EXTRACTION)
    dl = DataLoader(ds, batch_size=1, shuffle=False)

    model.eval()
    out = {}
    with torch.no_grad():
        for batch in tqdm(dl, desc="Extracting candidates", leave=False):
            ids = batch["ids"].to(config.DEVICE)
            mask = batch["mask"].to(config.DEVICE)
            orig = json.loads(batch["orig_data"][0])
            sid = orig["id"]

            logits = model(ids, mask)
            pred = torch.argmax(logits, dim=2).cpu().numpy()[0]
            aspects, opinions = decode_spans_from_tags(batch["ids"][0].cpu().numpy(), pred, tokenizer)
            out[sid] = {"aspects": aspects, "opinions": opinions}
    return out

# 7 — OOF Wrong Spans (INNER K-Fold)
def oof_wrong_spans(train_items: List[Dict[str, Any]], tokenizer: RobertaTokenizerFast) -> Dict[str, Dict[str, List[str]]]:
    inner_k = config.OOF_K
    kf = KFold(n_splits=inner_k, shuffle=True, random_state=config.SEED)

    wrong_map = {it["id"]: {"wrong_aspects": [], "wrong_opinions": []} for it in train_items}

    gold_aspects = {
        it["id"]: {norm_text(t["aspect"]) for t in it["triplets"] if not is_null(t.get("aspect"))}
        for it in train_items
    }
    gold_opinions = {
        it["id"]: {norm_text(t["opinion"]) for t in it["triplets"] if not is_null(t.get("opinion"))}
        for it in train_items
    }

    for split_i, (tr_idx, va_idx) in enumerate(kf.split(train_items), start=1):
        inner_train = [train_items[i] for i in tr_idx]
        inner_val = [train_items[i] for i in va_idx]

        print(f"\n   [OOF] Split {split_i}/{inner_k}: train={len(inner_train)} val={len(inner_val)}")
        extractor = train_extractor(inner_train, tokenizer, epochs=config.OOF_EPOCHS_EXTRACTION)
        cand = extract_candidates(extractor, inner_val, tokenizer)

        for it in inner_val:
            sid = it["id"]
            pred_asps = cand.get(sid, {}).get("aspects", [])
            pred_ops = cand.get(sid, {}).get("opinions", [])

            pred_asps = [a for a in pred_asps if a and not is_null(a)]
            pred_ops = [o for o in pred_ops if o and not is_null(o)]

            wa = [a for a in pred_asps if norm_text(a) not in gold_aspects[sid]]
            wo = [o for o in pred_ops if norm_text(o) not in gold_opinions[sid]]

            wa = wa[:config.MAX_WRONG_SPANS_PER_SENT]
            wo = wo[:config.MAX_WRONG_SPANS_PER_SENT]

            wrong_map[sid]["wrong_aspects"].extend(wa)
            wrong_map[sid]["wrong_opinions"].extend(wo)

    # Dedup
    for sid in wrong_map:
        wrong_map[sid]["wrong_aspects"] = list(dict.fromkeys([x for x in wrong_map[sid]["wrong_aspects"] if x]))
        wrong_map[sid]["wrong_opinions"] = list(dict.fromkeys([x for x in wrong_map[sid]["wrong_opinions"] if x]))

    return wrong_map

# 8 — Pair Dataset/Model
class PairDataset(Dataset):
    """
    VALID includes paired & unpaired gold.
    INVALID includes wrong pairs + unpaired wrong.
    """
    def __init__(
        self,
        items: List[Dict[str, Any]],
        tokenizer: RobertaTokenizerFast,
        max_len: int,
        wrong_map: Optional[Dict[str, Dict[str, List[str]]]] = None,
    ):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.examples = []

        INVALID = 0
        VALID = 1

        for item in items:
            sid = item["id"]
            text = safe_text(item.get("text", ""))

            gold_pairs = set((norm_text(t["aspect"]), norm_text(t["opinion"])) for t in item["triplets"])

            # Positives (paired + unpaired)
            for t in item["triplets"]:
                v, a = t.get("valence"), t.get("arousal")
                if v is None or a is None:
                    continue
                self.examples.append({
                    "sid": sid, "text": text,
                    "a": normalize_candidate_token(t.get("aspect")),
                    "o": normalize_candidate_token(t.get("opinion")),
                    "y": VALID,
                    "va": (float(v), float(a)),
                })

            # Wrong spans
            wrong_asps, wrong_ops = [], []
            if wrong_map is not None and sid in wrong_map:
                wrong_asps = wrong_map[sid].get("wrong_aspects", [])
                wrong_ops = wrong_map[sid].get("wrong_opinions", [])

            # Unique gold aspects/opinions (include NULL if in data)
            gold_aspects = uniq_keep_order([normalize_candidate_token(t.get("aspect")) for t in item["triplets"]])
            gold_opinions = uniq_keep_order([normalize_candidate_token(t.get("opinion")) for t in item["triplets"]])

            # Unpaired wrong (hard negatives)
            unpaired_wrong = []
            if config.ADD_UNPAIRED_FROM_WRONG:
                for a_ in wrong_asps:
                    unpaired_wrong.append((a_, "NULL"))
                for o_ in wrong_ops:
                    unpaired_wrong.append(("NULL", o_))

            # Dedup + cap
            unpaired_seen = set()
            unpaired_uniq = []
            for a_, o_ in unpaired_wrong:
                k = (norm_text(a_), norm_text(o_))
                if k in unpaired_seen:
                    continue
                unpaired_seen.add(k)
                unpaired_uniq.append((a_, o_))
                if len(unpaired_uniq) >= config.MAX_UNPAIRED_PER_SENT:
                    break

            for a_, o_ in unpaired_uniq:
                if (norm_text(a_), norm_text(o_)) in gold_pairs:
                    continue
                self.examples.append({
                    "sid": sid, "text": text, "a": a_, "o": o_,
                    "y": INVALID,
                    "va": (5.0, 5.0)
                })

            # Other negatives
            neg_pairs = []

            for a_ in wrong_asps:
                for o_ in gold_opinions:
                    neg_pairs.append((a_, o_))
            for a_ in gold_aspects:
                for o_ in wrong_ops:
                    neg_pairs.append((a_, o_))
            for a_ in wrong_asps:
                for o_ in wrong_ops:
                    neg_pairs.append((a_, o_))

            # cross product gaps (incl NULL if in gold lists), except true gold
            for a_ in gold_aspects:
                for o_ in gold_opinions:
                    if (norm_text(a_), norm_text(o_)) not in gold_pairs:
                        neg_pairs.append((a_, o_))

            uniq = []
            seen = set()
            for a_, o_ in neg_pairs:
                ka, ko = norm_text(a_), norm_text(o_)
                if not ka or not ko:
                    continue
                if (ka, ko) in gold_pairs:
                    continue
                if (ka, ko) in unpaired_seen:
                    continue
                if (ka, ko) in seen:
                    continue
                seen.add((ka, ko))
                uniq.append((a_, o_))
                if len(uniq) >= config.MAX_NEG_PAIRS_PER_SENT:
                    break

            for a_, o_ in uniq:
                self.examples.append({
                    "sid": sid, "text": text, "a": a_, "o": o_,
                    "y": INVALID,
                    "va": (5.0, 5.0)
                })

        # Final dedup
        seen = set()
        deduped = []
        for ex in self.examples:
            key = (
                norm_text(ex["text"]),
                norm_text(ex["a"]),
                norm_text(ex["o"]),
                round(float(ex["va"][0]), 6),
                round(float(ex["va"][1]), 6),
                int(ex["y"]),
            )
            if key in seen:
                continue
            seen.add(key)
            deduped.append(ex)
        self.examples = deduped

        num_valid = sum(1 for ex in self.examples if ex["y"] == 1)
        num_invalid = len(self.examples) - num_valid
        print("\nPair Dataset Stats")
        print(f"Total: {len(self.examples)} | Valid: {num_valid} | Invalid: {num_invalid}")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.examples[idx]
        text, a, o = ex["text"], ex["a"], ex["o"]

        if config.USE_NULL_TOKEN:
            s = f"{NULL_TOKEN} {text} </s></s> {a} </s></s> {o}"
        else:
            s = f"{text} </s></s> {a} </s></s> {o}"

        enc = self.tokenizer(
            s,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "ids": enc["input_ids"].squeeze(0),
            "mask": enc["attention_mask"].squeeze(0),
            "y": torch.tensor(ex["y"], dtype=torch.long),
            "va": torch.tensor(ex["va"], dtype=torch.float),
            "meta": json.dumps({"sid": ex["sid"], "a": a, "o": o}),
        }

class PairModel(nn.Module):
    def __init__(self, tokenizer: RobertaTokenizerFast):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(config.MODEL_NAME)
        self.encoder.resize_token_embeddings(len(tokenizer))
        self.drop = nn.Dropout(0.2)
        hs = self.encoder.config.hidden_size
        self.cls_head = nn.Linear(hs, 2)
        self.va_head = nn.Linear(hs, 2)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.drop(out.last_hidden_state[:, 0, :])
        logits = self.cls_head(cls)
        va = self.va_head(cls)
        return logits, va

def pair_loss(logits, va_pred, y, va_true) -> torch.Tensor:
    loss_cls = nn.CrossEntropyLoss()(logits, y)

    mask = (y == 1)
    if mask.any():
        pred = torch.clamp(va_pred[mask], 1.0, 9.0)
        tgt = va_true[mask]
        loss_va = nn.MSELoss()(pred, tgt)
    else:
        loss_va = torch.zeros((), device=logits.device)

    return loss_cls + loss_va

def train_pair_model(pair_items, tokenizer, wrong_map) -> PairModel:
    ds = PairDataset(pair_items, tokenizer, config.MAX_LEN_PAIR, wrong_map=wrong_map)
    dl = DataLoader(ds, batch_size=config.BATCH_SIZE_PAIR, shuffle=True)

    model = PairModel(tokenizer).to(config.DEVICE)
    opt = AdamW(model.parameters(), lr=config.LR, weight_decay=0.01)

    for epoch in range(config.MAX_EPOCHS_PAIR):
        model.train()
        tot = 0.0
        for batch in tqdm(dl, desc=f"PairModel Epoch {epoch+1}/{config.MAX_EPOCHS_PAIR}"):
            ids = batch["ids"].to(config.DEVICE)
            mask = batch["mask"].to(config.DEVICE)
            y = batch["y"].to(config.DEVICE)
            va = batch["va"].to(config.DEVICE)

            opt.zero_grad()
            logits, va_pred = model(ids, mask)
            loss = pair_loss(logits, va_pred, y, va)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += float(loss.item())

        print(f"   Epoch {epoch+1} | AvgLoss {tot/max(1,len(dl)):.4f}")

    return model

# 9 — Inference (NULL aware)
def infer_dimaste_triplet(
    extractor: ExtractionModel,
    pair_model: PairModel,
    items: List[Dict[str, Any]],
    tokenizer: RobertaTokenizerFast,
) -> Tuple[List[List[Dict[str, Any]]], List[List[Dict[str, Any]]], List[Dict[str, Any]]]:

    cand = extract_candidates(extractor, items, tokenizer)

    preds_per_sent: List[List[Dict[str, Any]]] = []
    golds_per_sent: List[List[Dict[str, Any]]] = []
    rows: List[Dict[str, Any]] = []

    pair_model.eval()
    with torch.no_grad():
        for it in tqdm(items, desc="Inferring DimASTE", leave=False):
            sid = it["id"]
            text = safe_text(it.get("text", ""))

            # Gold includes paired & unpaired as long as VA exists
            golds = [{
                "aspect": normalize_candidate_token(t["aspect"]),
                "opinion": normalize_candidate_token(t["opinion"]),
                "valence": t["valence"],
                "arousal": t["arousal"]
            } for t in it.get("triplets", [])
              if (t.get("valence") is not None and t.get("arousal") is not None)]
            golds_per_sent.append(golds)

            aspects = [normalize_candidate_token(a) for a in cand.get(sid, {}).get("aspects", [])]
            opinions = [normalize_candidate_token(o) for o in cand.get(sid, {}).get("opinions", [])]

            aspects = [a for a in aspects if a and not is_null(a)]
            opinions = [o for o in opinions if o and not is_null(o)]

            sent_preds: List[Dict[str, Any]] = []

            def score_pair(a_: str, o_: str) -> Optional[Dict[str, Any]]:
                if config.USE_NULL_TOKEN:
                    s = f"{NULL_TOKEN} {text} </s></s> {a_} </s></s> {o_}"
                else:
                    s = f"{text} </s></s> {a_} </s></s> {o_}"

                enc = tokenizer(
                    s,
                    padding="max_length",
                    truncation=True,
                    max_length=config.MAX_LEN_PAIR,
                    return_tensors="pt"
                )
                ids = enc["input_ids"].to(config.DEVICE)
                mask = enc["attention_mask"].to(config.DEVICE)

                logits, va = pair_model(ids, mask)
                y = int(torch.argmax(logits, dim=1).item())
                if y == 0:
                    return None

                va = va.squeeze(0).cpu().numpy()
                v = float(np.clip(va[0], 1.0, 9.0))
                ar = float(np.clip(va[1], 1.0, 9.0))

                return {
                    "aspect": a_,
                    "opinion": o_,
                    "valence": float(f"{v:.2f}"),
                    "arousal": float(f"{ar:.2f}")
                }

            if aspects and opinions:
                for a in aspects:
                    for o in opinions:
                        p = score_pair(a, o)
                        if p is not None:
                            sent_preds.append(p)
            elif opinions and not aspects:
                for o in opinions:
                    p = score_pair("NULL", o)
                    if p is not None:
                        sent_preds.append(p)
            elif aspects and not opinions:
                for a in aspects:
                    p = score_pair(a, "NULL")
                    if p is not None:
                        sent_preds.append(p)
            else:
                p = score_pair("NULL", "NULL")
                if p is not None:
                    sent_preds.append(p)

            uniq = {}
            for p in sent_preds:
                k = (norm_text(p["aspect"]), norm_text(p["opinion"]))
                if k not in uniq:
                    uniq[k] = p
            sent_preds = list(uniq.values())

            preds_per_sent.append(sent_preds)

            if sent_preds:
                for p in sent_preds:
                    rows.append({
                        "Sentence_ID": sid,
                        "Text": text,
                        "Predicted_Aspect": p["aspect"],
                        "Predicted_Opinion": p["opinion"],
                        "Predicted_Valence": p["valence"],
                        "Predicted_Arousal": p["arousal"],
                    })
            else:
                rows.append({
                    "Sentence_ID": sid,
                    "Text": text,
                    "Predicted_Aspect": "NULL",
                    "Predicted_Opinion": "NULL",
                    "Predicted_Valence": 5.00,
                    "Predicted_Arousal": 5.00,
                })

    return preds_per_sent, golds_per_sent, rows

# 10 — Metrics
def dimaste_triplet_metrics(preds_per_sent: List[List[Dict[str, Any]]], golds_per_sent: List[List[Dict[str, Any]]]) -> Dict[str, float]:
    TP = FP = FN = 0
    total_dist = 0.0
    matched_va_pairs = []

    for preds, golds in zip(preds_per_sent, golds_per_sent):
        matched = set()
        for p in preds:
            ok = False
            for gi, g in enumerate(golds):
                if gi in matched:
                    continue
                if norm_text(p["aspect"]) == norm_text(g["aspect"]) and norm_text(p["opinion"]) == norm_text(g["opinion"]):
                    TP += 1
                    matched.add(gi)
                    ok = True
                    dist = calculate_normalized_euclidean_distance(p["valence"], p["arousal"], g["valence"], g["arousal"])
                    total_dist += dist
                    matched_va_pairs.append((g["valence"], p["valence"], g["arousal"], p["arousal"]))
                    break
            if not ok:
                FP += 1
        FN += len(golds) - len(matched)

    cTP = TP - total_dist
    cRecall = cTP / (TP + FN) if (TP + FN) > 0 else 0.0
    cPrecision = cTP / (TP + FP) if (TP + FP) > 0 else 0.0
    cF1 = 2 * cRecall * cPrecision / (cRecall + cPrecision) if (cRecall + cPrecision) > 0 else 0.0

    mae_v = mae_a = pv = pa = 0.0
    if len(matched_va_pairs) > 1:
        gv = np.array([x[0] for x in matched_va_pairs], dtype=float)
        pv_ = np.array([x[1] for x in matched_va_pairs], dtype=float)
        ga = np.array([x[2] for x in matched_va_pairs], dtype=float)
        pa_ = np.array([x[3] for x in matched_va_pairs], dtype=float)
        mae_v = float(np.mean(np.abs(gv - pv_)))
        mae_a = float(np.mean(np.abs(ga - pa_)))
        try:
            pv, _ = pearsonr(gv, pv_)
            pa, _ = pearsonr(ga, pa_)
        except Exception:
            pv, pa = 0.0, 0.0

    return {
        "cRecall": float(cRecall),
        "cPrecision": float(cPrecision),
        "cF1": float(cF1),
        "TP_cat": int(TP),
        "FP_cat": int(FP),
        "FN_cat": int(FN),
        "VA_Error_Dist": float(total_dist),
        "MAE_V": float(mae_v),
        "MAE_A": float(mae_a),
        "Pearson_V": float(pv),
        "Pearson_A": float(pa),
    }

# 11 — Main (loop domains)
def run_for_domain(domain: str):
    t0 = time.time()
    print(f"\nRUN DOMAIN: {domain}")

    url = config.get_train_url(domain)
    filename = config.get_local_filename(domain)
    full_data = download_and_load(url, filename)
    if not full_data:
        print("No data loaded.")
        return

    cleaned_data, stats = cleaning_with_stats(full_data)

    trainable_full = filter_trainable_items(cleaned_data)
    trainable_ids = {it["id"] for it in trainable_full}
    non_trainable_items = [it for it in cleaned_data if it["id"] not in trainable_ids]

    print("\nData Separation Summary")
    print(f"Total after cleaning           : {len(cleaned_data)}")
    print(f"Trainable (has VA triplets)    : {len(trainable_full)}")
    print(f"Non-trainable (no VA triplets) : {len(non_trainable_items)}")

    tokenizer = RobertaTokenizerFast.from_pretrained(config.MODEL_NAME)
    if config.USE_NULL_TOKEN:
        if tokenizer.convert_tokens_to_ids(NULL_TOKEN) == tokenizer.unk_token_id:
            tokenizer.add_special_tokens({"additional_special_tokens": [NULL_TOKEN]})
        print("NULL token ready. vocab_size =", len(tokenizer), "| NULL_ID =", tokenizer.convert_tokens_to_ids(NULL_TOKEN))

    K = config.OUTER_K
    kf = KFold(n_splits=K, shuffle=True, random_state=config.SEED)

    all_fold_metrics = []
    all_pred_rows = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(trainable_full), start=1):
        fold_t0 = time.time()

        print(f"\nOUTER FOLD {fold}/{K} ({domain})\n")

        train_data = [trainable_full[i] for i in train_idx]
        test_data = [trainable_full[i] for i in test_idx]

        print(f"TRAIN: {len(train_data)} | TEST: {len(test_data)} | NON-TRAINABLE(extra): {len(non_trainable_items)}")
        print(f"Epochs now: extractor={config.MAX_EPOCHS_EXTRACTION}, pair={config.MAX_EPOCHS_PAIR}, inner_oof_extractor={config.OOF_EPOCHS_EXTRACTION}")
        print(f"Folds now: outer={config.OUTER_K}, inner_oof={config.OOF_K}")

        print("\n[A] Building OOF wrong spans for negatives (INNER KFold)")
        wrong_map = oof_wrong_spans(train_data, tokenizer)

        print("\n[B] Training extractor")
        extractor = train_extractor(train_data, tokenizer, epochs=config.MAX_EPOCHS_EXTRACTION)

        print("\n[C] Training pair model")
        pair_model = train_pair_model(train_data, tokenizer, wrong_map)

        print("\n[D1] Inference on test fold for METRICS")
        preds_t, golds_t, rows_t = infer_dimaste_triplet(extractor, pair_model, test_data, tokenizer)
        m = dimaste_triplet_metrics(preds_t, golds_t)
        all_fold_metrics.append({"Domain": domain, "Fold": fold, **m})

        for r in rows_t:
            r["Domain"] = domain
            r["Fold"] = fold
            r["Split"] = "test_trainable"
        all_pred_rows.extend(rows_t)

        if non_trainable_items:
            print("\n[D2] Extra inference on non-trainable items (no metrics)")
            _, _, rows_nt = infer_dimaste_triplet(extractor, pair_model, non_trainable_items, tokenizer)
            for r in rows_nt:
                r["Domain"] = domain
                r["Fold"] = fold
                r["Split"] = "non_trainable_only"
            all_pred_rows.extend(rows_nt)

        fold_dt = time.time() - fold_t0
        print(f"\nFold {fold} | cF1={m['cF1']:.4f} cRecall={m['cRecall']:.4f} cPrecision={m['cPrecision']:.4f} MAE_V={m['MAE_V']:.4f}")
        print(f"Fold runtime: {format_duration(fold_dt)}")

    pred_path = config.output_predictions_csv(domain)
    metrics_path = config.output_metrics_csv(domain)

    if all_pred_rows:
        dfp = pd.DataFrame(all_pred_rows)
        dfp.to_csv(pred_path, index=False, float_format="%.4f")

    dff = pd.DataFrame(all_fold_metrics)
    metric_cols = [c for c in dff.columns if c not in {"Domain", "Fold"}]

    mean = dff[metric_cols].mean(numeric_only=True).to_dict()
    std = dff[metric_cols].std(numeric_only=True).to_dict()

    dff2 = pd.concat([
        dff,
        pd.DataFrame([{"Domain": domain, "Fold": "MEAN", **mean}]),
        pd.DataFrame([{"Domain": domain, "Fold": "STD", **std}]),
    ], ignore_index=True)

    dff2.to_csv(metrics_path, index=False, float_format="%.4f")

    print("\nFINAL (MEAN ± STD) —", domain)
    print(f"cF1:        {mean.get('cF1',0):.4f} ± {std.get('cF1',0):.4f}")
    print(f"cRecall:    {mean.get('cRecall',0):.4f} ± {std.get('cRecall',0):.4f}")
    print(f"cPrecision: {mean.get('cPrecision',0):.4f} ± {std.get('cPrecision',0):.4f}")
    print(f"MAE_V:      {mean.get('MAE_V',0):.4f} ± {std.get('MAE_V',0):.4f}")
    print(f"MAE_A:      {mean.get('MAE_A',0):.4f} ± {std.get('MAE_A',0):.4f}")

    dt = time.time() - t0

    print(f"DOMAIN {domain} RUNTIME:", format_duration(dt))

def main():
    set_seed(config.SEED)
    t0 = time.time()

    for domain in config.DOMAINS:
        run_for_domain(domain)

    dt = time.time() - t0
    print("TOTAL RUNTIME (ALL DOMAINS):", format_duration(dt))

if __name__ == "__main__":
    main()

2025-12-11 19:08:52.691921: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765480132.909156      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765480132.975168      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

PairModel Epoch 2/3: 100%|██████████| 511/511 [12:41<00:00,  1.49s/it]


   Epoch 2 | AvgLoss 17.3145


PairModel Epoch 3/3: 100%|██████████| 511/511 [12:41<00:00,  1.49s/it]


   Epoch 3 | AvgLoss 4.8478

[D1] Inference on test fold for METRICS...



Fold 5 | cF1=0.4470 cRecall=0.4154 cPrecision=0.4838 MAE_V=0.4694
Fold runtime: 2h 2m 10s

✅ Saved predictions to dimaste_predictions_output_laptop.csv
✅ Saved metrics to dimaste_kfold_metrics_laptop.csv

FINAL (MEAN ± STD) — laptop
cF1:        0.3540 ± 0.1419
cRecall:    0.3110 ± 0.1208
cPrecision: 0.4143 ± 0.1786
MAE_V:      2.3870 ± 2.5964
MAE_A:      1.5867 ± 2.4319

DOMAIN laptop RUNTIME: 10h 9m 3s

TOTAL RUNTIME (ALL DOMAINS): 10h 9m 3s


# Restoran

In [ ]:
# 2 — Config
class Config:
    # Task
    SUBTASK = "subtask_2"
    LANG = "eng"
    DOMAINS = ["restaurant"]

    # Model
    MODEL_NAME = "roberta-base"
    MAX_LEN_EXTRACTION = 512
    MAX_LEN_PAIR = 512

    # Train
    BATCH_SIZE_EXTRACTION = 8
    BATCH_SIZE_PAIR = 16
    MAX_EPOCHS_EXTRACTION = 3
    MAX_EPOCHS_PAIR = 3
    LR = 2e-5
    SEED = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # OOF negatives
    OOF_K = 5
    OOF_EPOCHS_EXTRACTION = 3
    MAX_WRONG_SPANS_PER_SENT = 3
    MAX_NEG_PAIRS_PER_SENT = 10

    # NULL handling
    USE_NULL_TOKEN = True
    NULL_TOKEN = "[NULL]"

    # add hard unpaired negatives from wrong spans
    ADD_UNPAIRED_FROM_WRONG = True
    MAX_UNPAIRED_PER_SENT = 8

    # Outer fold
    OUTER_K = 5

    @classmethod
    def get_train_url(cls, domain: str) -> str:
        filename = f"{cls.LANG}_{domain}_train_alltasks.jsonl"
        return (
            "https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/"
            f"task-dataset/track_a/{cls.SUBTASK}/{cls.LANG}/{filename}"
        )

    @classmethod
    def get_local_filename(cls, domain: str) -> str:
        return f"{cls.LANG}_{domain}_train_alltasks.jsonl"

    @classmethod
    def output_metrics_csv(cls, domain: str) -> str:
        return f"dimaste_kfold_metrics_{domain}.csv"

    @classmethod
    def output_predictions_csv(cls, domain: str) -> str:
        return f"dimaste_predictions_output_{domain}.csv"

config = Config()

TAG2IDX = {'O': 0, 'B-ASP': 1, 'I-ASP': 2, 'B-OP': 3, 'I-OP': 4}
IDX2TAG = {v: k for k, v in TAG2IDX.items()}

D_MAX = np.sqrt(8**2 + 8**2)
NULL_TOKEN = config.NULL_TOKEN

print(f"Running on device: {config.DEVICE}")
print(f"Domains: {config.DOMAINS}")
print(f"NULL token enabled: {config.USE_NULL_TOKEN}")
print(f"Epochs: extractor={config.MAX_EPOCHS_EXTRACTION}, pair={config.MAX_EPOCHS_PAIR}, oof_extractor={config.OOF_EPOCHS_EXTRACTION}")
print(f"Folds: outer={config.OUTER_K}, inner_oof={config.OOF_K}")

# 3 — Seed & Helpers
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.SEED)

def format_duration(seconds: float) -> str:
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

def safe_text(x: Any) -> str:
    return x if isinstance(x, str) else ""

def norm_text(s: Any) -> str:
    s = safe_text(s)
    return re.sub(r"\s+", " ", s.strip().lower())

def is_null(x: Any) -> bool:
    if x is None:
        return True
    if isinstance(x, str):
        t = x.strip()
        if t.upper() == "NULL":
            return True
        if config.USE_NULL_TOKEN and t == NULL_TOKEN:
            return True
    return False

def normalize_candidate_token(x: Any) -> str:
    # Map internal [NULL] token to dataset-style "NULL"
    if isinstance(x, str) and x.strip() == NULL_TOKEN:
        return "NULL"
    return safe_text(x)

def dedup_triplets_exact(triplets: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for t in triplets:
        a = t.get("aspect")
        o = t.get("opinion")
        v = t.get("valence")
        ar = t.get("arousal")
        key = (
            norm_text(a) if isinstance(a, str) else str(a),
            norm_text(o) if isinstance(o, str) else str(o),
            round(float(v), 6) if v is not None else None,
            round(float(ar), 6) if ar is not None else None,
        )
        if key in seen:
            continue
        seen.add(key)
        out.append(t)
    return out

def calculate_normalized_euclidean_distance(vp, ap, vg, ag) -> float:
    dist = np.sqrt((vp - vg)**2 + (ap - ag)**2) / D_MAX
    return float(np.clip(dist, 0.0, 1.0))

def find_all_spans(text: str, phrase: str) -> List[Tuple[int, int]]:
    if not phrase:
        return []
    return [(m.start(), m.end()) for m in re.finditer(re.escape(phrase), text)]

def pick_closest_pair(
    asp_spans: List[Tuple[int, int]],
    op_spans: List[Tuple[int, int]]
) -> Optional[Tuple[int, int, int, int]]:
    best = None
    best_dist = 10**18
    for a_s, a_e in asp_spans:
        a_c = (a_s + a_e) / 2
        for o_s, o_e in op_spans:
            o_c = (o_s + o_e) / 2
            d = abs(a_c - o_c)
            if d < best_dist:
                best_dist = d
                best = (a_s, a_e, o_s, o_e)
    return best

def roberta_tokens_to_text(tokens: List[str]) -> str:
    out = ""
    for t in tokens:
        if t in ["<s>", "</s>", "<pad>"]:
            continue
        if t.startswith("Ġ"):
            out += (" " + t[1:]) if out else t[1:]
        else:
            out += t
    return out.strip()

def uniq_keep_order(xs: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in xs:
        k = norm_text(x)
        if not k:
            continue
        if k in seen:
            continue
        seen.add(k)
        out.append(x)
    return out

# 4 — Download & Load (with retry)
def download_and_load(url: str, filename: str) -> List[Dict[str, Any]]:
    if not os.path.exists(filename):
        last_err = None
        for attempt in range(3):
            try:
                r = requests.get(url, timeout=60)
                r.raise_for_status()
                with open(filename, "wb") as f:
                    f.write(r.content)
                break
            except Exception as e:
                last_err = e
                print(f"  download attempt {attempt+1}/3 failed: {e}")
                time.sleep(2 + attempt)
        else:
            raise RuntimeError(f"Failed downloading after retries: {last_err}")

    data = []
    total_triplets_initial = 0
    total_triplets_deduped = 0

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            entry = json.loads(line)

            triplets = []
            for quad in entry.get("Quadruplet", []):
                asp = quad.get("Aspect", "NULL")
                op = quad.get("Opinion", "NULL")

                v = a = None
                va_raw = quad.get("VA", None)
                if isinstance(va_raw, str) and "#" in va_raw:
                    try:
                        v, a = map(float, va_raw.split("#"))
                    except Exception:
                        v, a = None, None

                triplets.append({
                    "aspect": asp,
                    "opinion": op,
                    "valence": v,
                    "arousal": a,
                })

            total_triplets_initial += len(triplets)
            triplets = dedup_triplets_exact(triplets)
            total_triplets_deduped += len(triplets)

            data.append({
                "id": entry.get("ID"),
                "text": entry.get("Text", ""),
                "triplets": triplets,
            })

    print("\nLoad Summary")
    print(f"Items                 : {len(data)}")
    print(f"Triplets (initial)    : {total_triplets_initial}")
    print(f"Triplets (deduped)    : {total_triplets_deduped}")
    return data

# 5 — Cleaning + Stats
def cleaning_with_stats(data: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    original = len(data)

    texts = [norm_text(x.get("text", "")) for x in data]
    counts = {}
    for t in texts:
        counts[t] = counts.get(t, 0) + 1
    dup_total = sum(c - 1 for c in counts.values() if c > 1)

    # Remove duplicates (keep first)
    seen = set()
    deduped = []
    for it in data:
        t = norm_text(it.get("text", ""))
        if t in seen:
            continue
        seen.add(t)
        deduped.append(it)
    after_dedup = len(deduped)

    # Missing/empty definition: invalid text OR empty triplets list
    missing_or_empty = 0
    cleaned = []
    for it in deduped:
        t = safe_text(it.get("text", ""))
        invalid_text = (
            (t is None) or
            (t == "") or
            (t.strip() == "") or
            (t.strip().upper() in {"NULL", "NONE"}) or
            (t.strip().lower() == "nan")
        )
        empty_triplets = not isinstance(it.get("triplets", []), list) or len(it.get("triplets", [])) == 0

        if invalid_text or empty_triplets:
            missing_or_empty += 1
            continue
        cleaned.append(it)

    after_remove_missing = len(cleaned)

    print("\nDATA CLEANING STATS")
    print(f"Jumlah data asli                                 : {original}")
    print(f"Jumlah data duplikat (berdasarkan Text)           : {dup_total}")
    print(f"Jumlah data setelah hapus duplikat                : {after_dedup}")
    print(f"Jumlah data missing/kosong (Text invalid/Triplet empty) : {missing_or_empty}")
    print(f"Jumlah data setelah hapus missing/kosong          : {after_remove_missing}")

    stats = {
        "original": original,
        "duplicates": dup_total,
        "after_dedup": after_dedup,
        "missing_or_empty": missing_or_empty,
        "after_remove_missing": after_remove_missing,
    }
    return cleaned, stats

def filter_trainable_items(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Trainable = punya minimal 1 triplet dengan VA valid.
    Keep paired & unpaired (NULL) as long as VA exists.
    """
    out = []
    for it in items:
        good = []
        for t in it.get("triplets", []):
            v = t.get("valence")
            a = t.get("arousal")
            if v is None or a is None:
                continue
            good.append({
                "aspect": normalize_candidate_token(t.get("aspect")),
                "opinion": normalize_candidate_token(t.get("opinion")),
                "valence": float(v),
                "arousal": float(a),
            })
        if good:
            out.append({"id": it["id"], "text": it["text"], "triplets": good})
    return out

# 6 — Extraction Dataset/Model (BIO)
class ExtractionDataset(Dataset):
    def __init__(self, data: List[Dict[str, Any]], tokenizer: RobertaTokenizerFast, max_len: int):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.special_ids = {tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = self.data[idx]
        text = safe_text(item.get("text", ""))

        if config.USE_NULL_TOKEN:
            aug_text = f"{NULL_TOKEN} {text}"
            prefix_len = len(NULL_TOKEN) + 1
        else:
            aug_text = text
            prefix_len = 0

        enc = self.tokenizer(
            aug_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        input_ids = enc["input_ids"].squeeze(0)
        mask = enc["attention_mask"].squeeze(0)
        offsets = enc["offset_mapping"].squeeze(0).tolist()

        labels = torch.zeros(self.max_len, dtype=torch.long)
        for i in range(self.max_len):
            if int(input_ids[i].item()) in self.special_ids:
                labels[i] = -100

        def overlap(start, end, s, e):
            return (start < e) and (end > s) and (start < end)

        # Label only non-NULL spans (NULL is not a span in text)
        for t in item.get("triplets", []):
            asp = t.get("aspect")
            op = t.get("opinion")
            if is_null(asp) or is_null(op):
                continue

            asp = safe_text(asp)
            op = safe_text(op)

            asp_spans = find_all_spans(text, asp)
            op_spans = find_all_spans(text, op)
            if not asp_spans or not op_spans:
                continue

            best = pick_closest_pair(asp_spans, op_spans)
            if best is None:
                continue
            asp_s, asp_e, op_s, op_e = best

            asp_s += prefix_len
            asp_e += prefix_len
            op_s += prefix_len
            op_e += prefix_len

            asp_first, op_first = True, True
            for i, (st, en) in enumerate(offsets):
                if i >= self.max_len:
                    break
                if int(input_ids[i].item()) in self.special_ids:
                    continue

                if overlap(st, en, asp_s, asp_e):
                    labels[i] = TAG2IDX["B-ASP"] if asp_first else TAG2IDX["I-ASP"]
                    asp_first = False

                if overlap(st, en, op_s, op_e):
                    labels[i] = TAG2IDX["B-OP"] if op_first else TAG2IDX["I-OP"]
                    op_first = False

        return {
            "ids": input_ids,
            "mask": mask,
            "labels": labels,
            "orig_data": json.dumps(item),
        }

class ExtractionModel(nn.Module):
    def __init__(self, n_tags: int, tokenizer: RobertaTokenizerFast):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(config.MODEL_NAME)
        self.encoder.resize_token_embeddings(len(tokenizer))
        self.drop = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, n_tags)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        seq = self.drop(out.last_hidden_state)
        return self.classifier(seq)

def loss_extraction(logits, labels) -> torch.Tensor:
    return nn.CrossEntropyLoss(ignore_index=-100)(
        logits.view(-1, len(TAG2IDX)),
        labels.view(-1)
    )

def decode_spans_from_tags(
    input_ids: np.ndarray,
    tag_preds: np.ndarray,
    tokenizer: RobertaTokenizerFast
) -> Tuple[List[str], List[str]]:
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    SPECIAL = {tokenizer.bos_token, tokenizer.eos_token, tokenizer.pad_token}

    aspects, opinions = [], []
    current = None

    for tok, tidx in zip(tokens, tag_preds):
        if tok in SPECIAL:
            if current:
                (aspects if current["type"] == "ASP" else opinions).append(current)
                current = None
            continue

        tag = IDX2TAG.get(int(tidx), "O")
        if tag.startswith("B-"):
            if current:
                (aspects if current["type"] == "ASP" else opinions).append(current)
            current_type = "ASP" if "ASP" in tag else "OP"
            current = {"type": current_type, "raw_tokens": [tok]}
        elif tag.startswith("I-") and current:
            if ("ASP" in tag and current["type"] == "ASP") or ("OP" in tag and current["type"] == "OP"):
                current["raw_tokens"].append(tok)
            else:
                (aspects if current["type"] == "ASP" else opinions).append(current)
                current = None
        else:
            if current:
                (aspects if current["type"] == "ASP" else opinions).append(current)
                current = None

    if current:
        (aspects if current["type"] == "ASP" else opinions).append(current)

    asp_txt = [roberta_tokens_to_text(x["raw_tokens"]) for x in aspects]
    op_txt = [roberta_tokens_to_text(x["raw_tokens"]) for x in opinions]

    asp_txt = uniq_keep_order([normalize_candidate_token(x) for x in asp_txt if x])
    op_txt = uniq_keep_order([normalize_candidate_token(x) for x in op_txt if x])
    return asp_txt, op_txt

def train_extractor(train_data, tokenizer, epochs: int) -> ExtractionModel:
    ds = ExtractionDataset(train_data, tokenizer, config.MAX_LEN_EXTRACTION)
    dl = DataLoader(ds, batch_size=config.BATCH_SIZE_EXTRACTION, shuffle=True)

    model = ExtractionModel(len(TAG2IDX), tokenizer).to(config.DEVICE)
    opt = AdamW(model.parameters(), lr=config.LR, weight_decay=0.01)

    for epoch in range(epochs):
        model.train()
        tot = 0.0
        for batch in tqdm(dl, desc=f"Extractor Epoch {epoch+1}/{epochs}"):
            ids = batch["ids"].to(config.DEVICE)
            mask = batch["mask"].to(config.DEVICE)
            labels = batch["labels"].to(config.DEVICE)

            opt.zero_grad()
            logits = model(ids, mask)
            loss = loss_extraction(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += float(loss.item())

        print(f"   Epoch {epoch+1} | AvgLoss {tot/max(1,len(dl)):.4f}")
    return model

def extract_candidates(
    model: ExtractionModel,
    data: List[Dict[str, Any]],
    tokenizer: RobertaTokenizerFast
) -> Dict[str, Dict[str, List[str]]]:
    ds = ExtractionDataset(data, tokenizer, config.MAX_LEN_EXTRACTION)
    dl = DataLoader(ds, batch_size=1, shuffle=False)

    model.eval()
    out = {}
    with torch.no_grad():
        for batch in tqdm(dl, desc="Extracting candidates", leave=False):
            ids = batch["ids"].to(config.DEVICE)
            mask = batch["mask"].to(config.DEVICE)
            orig = json.loads(batch["orig_data"][0])
            sid = orig["id"]

            logits = model(ids, mask)
            pred = torch.argmax(logits, dim=2).cpu().numpy()[0]
            aspects, opinions = decode_spans_from_tags(batch["ids"][0].cpu().numpy(), pred, tokenizer)
            out[sid] = {"aspects": aspects, "opinions": opinions}
    return out

# 7 — OOF Wrong Spans (INNER K-Fold)
def oof_wrong_spans(train_items: List[Dict[str, Any]], tokenizer: RobertaTokenizerFast) -> Dict[str, Dict[str, List[str]]]:
    inner_k = config.OOF_K
    kf = KFold(n_splits=inner_k, shuffle=True, random_state=config.SEED)

    wrong_map = {it["id"]: {"wrong_aspects": [], "wrong_opinions": []} for it in train_items}

    gold_aspects = {
        it["id"]: {norm_text(t["aspect"]) for t in it["triplets"] if not is_null(t.get("aspect"))}
        for it in train_items
    }
    gold_opinions = {
        it["id"]: {norm_text(t["opinion"]) for t in it["triplets"] if not is_null(t.get("opinion"))}
        for it in train_items
    }

    for split_i, (tr_idx, va_idx) in enumerate(kf.split(train_items), start=1):
        inner_train = [train_items[i] for i in tr_idx]
        inner_val = [train_items[i] for i in va_idx]

        print(f"\n   [OOF] Split {split_i}/{inner_k}: train={len(inner_train)} val={len(inner_val)}")
        extractor = train_extractor(inner_train, tokenizer, epochs=config.OOF_EPOCHS_EXTRACTION)
        cand = extract_candidates(extractor, inner_val, tokenizer)

        for it in inner_val:
            sid = it["id"]
            pred_asps = cand.get(sid, {}).get("aspects", [])
            pred_ops = cand.get(sid, {}).get("opinions", [])

            pred_asps = [a for a in pred_asps if a and not is_null(a)]
            pred_ops = [o for o in pred_ops if o and not is_null(o)]

            wa = [a for a in pred_asps if norm_text(a) not in gold_aspects[sid]]
            wo = [o for o in pred_ops if norm_text(o) not in gold_opinions[sid]]

            wa = wa[:config.MAX_WRONG_SPANS_PER_SENT]
            wo = wo[:config.MAX_WRONG_SPANS_PER_SENT]

            wrong_map[sid]["wrong_aspects"].extend(wa)
            wrong_map[sid]["wrong_opinions"].extend(wo)

    # Dedup
    for sid in wrong_map:
        wrong_map[sid]["wrong_aspects"] = list(dict.fromkeys([x for x in wrong_map[sid]["wrong_aspects"] if x]))
        wrong_map[sid]["wrong_opinions"] = list(dict.fromkeys([x for x in wrong_map[sid]["wrong_opinions"] if x]))

    return wrong_map

# 8 — Pair Dataset/Model
class PairDataset(Dataset):
    """
    VALID includes paired & unpaired gold.
    INVALID includes wrong pairs + unpaired wrong.
    """
    def __init__(
        self,
        items: List[Dict[str, Any]],
        tokenizer: RobertaTokenizerFast,
        max_len: int,
        wrong_map: Optional[Dict[str, Dict[str, List[str]]]] = None,
    ):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.examples = []

        INVALID = 0
        VALID = 1

        for item in items:
            sid = item["id"]
            text = safe_text(item.get("text", ""))

            gold_pairs = set((norm_text(t["aspect"]), norm_text(t["opinion"])) for t in item["triplets"])

            # Positives (paired + unpaired)
            for t in item["triplets"]:
                v, a = t.get("valence"), t.get("arousal")
                if v is None or a is None:
                    continue
                self.examples.append({
                    "sid": sid, "text": text,
                    "a": normalize_candidate_token(t.get("aspect")),
                    "o": normalize_candidate_token(t.get("opinion")),
                    "y": VALID,
                    "va": (float(v), float(a)),
                })

            # Wrong spans
            wrong_asps, wrong_ops = [], []
            if wrong_map is not None and sid in wrong_map:
                wrong_asps = wrong_map[sid].get("wrong_aspects", [])
                wrong_ops = wrong_map[sid].get("wrong_opinions", [])

            # Unique gold aspects/opinions (include NULL if in data)
            gold_aspects = uniq_keep_order([normalize_candidate_token(t.get("aspect")) for t in item["triplets"]])
            gold_opinions = uniq_keep_order([normalize_candidate_token(t.get("opinion")) for t in item["triplets"]])

            # Unpaired wrong (hard negatives)
            unpaired_wrong = []
            if config.ADD_UNPAIRED_FROM_WRONG:
                for a_ in wrong_asps:
                    unpaired_wrong.append((a_, "NULL"))
                for o_ in wrong_ops:
                    unpaired_wrong.append(("NULL", o_))

            # Dedup + cap
            unpaired_seen = set()
            unpaired_uniq = []
            for a_, o_ in unpaired_wrong:
                k = (norm_text(a_), norm_text(o_))
                if k in unpaired_seen:
                    continue
                unpaired_seen.add(k)
                unpaired_uniq.append((a_, o_))
                if len(unpaired_uniq) >= config.MAX_UNPAIRED_PER_SENT:
                    break

            for a_, o_ in unpaired_uniq:
                if (norm_text(a_), norm_text(o_)) in gold_pairs:
                    continue
                self.examples.append({
                    "sid": sid, "text": text, "a": a_, "o": o_,
                    "y": INVALID,
                    "va": (5.0, 5.0)
                })

            # Other negatives
            neg_pairs = []

            for a_ in wrong_asps:
                for o_ in gold_opinions:
                    neg_pairs.append((a_, o_))
            for a_ in gold_aspects:
                for o_ in wrong_ops:
                    neg_pairs.append((a_, o_))
            for a_ in wrong_asps:
                for o_ in wrong_ops:
                    neg_pairs.append((a_, o_))

            # cross product gaps (incl NULL if in gold lists), except true gold
            for a_ in gold_aspects:
                for o_ in gold_opinions:
                    if (norm_text(a_), norm_text(o_)) not in gold_pairs:
                        neg_pairs.append((a_, o_))

            uniq = []
            seen = set()
            for a_, o_ in neg_pairs:
                ka, ko = norm_text(a_), norm_text(o_)
                if not ka or not ko:
                    continue
                if (ka, ko) in gold_pairs:
                    continue
                if (ka, ko) in unpaired_seen:
                    continue
                if (ka, ko) in seen:
                    continue
                seen.add((ka, ko))
                uniq.append((a_, o_))
                if len(uniq) >= config.MAX_NEG_PAIRS_PER_SENT:
                    break

            for a_, o_ in uniq:
                self.examples.append({
                    "sid": sid, "text": text, "a": a_, "o": o_,
                    "y": INVALID,
                    "va": (5.0, 5.0)
                })

        # Final dedup
        seen = set()
        deduped = []
        for ex in self.examples:
            key = (
                norm_text(ex["text"]),
                norm_text(ex["a"]),
                norm_text(ex["o"]),
                round(float(ex["va"][0]), 6),
                round(float(ex["va"][1]), 6),
                int(ex["y"]),
            )
            if key in seen:
                continue
            seen.add(key)
            deduped.append(ex)
        self.examples = deduped

        num_valid = sum(1 for ex in self.examples if ex["y"] == 1)
        num_invalid = len(self.examples) - num_valid
        print("\nPair Dataset Stats")
        print(f"Total: {len(self.examples)} | Valid: {num_valid} | Invalid: {num_invalid}")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.examples[idx]
        text, a, o = ex["text"], ex["a"], ex["o"]

        if config.USE_NULL_TOKEN:
            s = f"{NULL_TOKEN} {text} </s></s> {a} </s></s> {o}"
        else:
            s = f"{text} </s></s> {a} </s></s> {o}"

        enc = self.tokenizer(
            s,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "ids": enc["input_ids"].squeeze(0),
            "mask": enc["attention_mask"].squeeze(0),
            "y": torch.tensor(ex["y"], dtype=torch.long),
            "va": torch.tensor(ex["va"], dtype=torch.float),
            "meta": json.dumps({"sid": ex["sid"], "a": a, "o": o}),
        }

class PairModel(nn.Module):
    def __init__(self, tokenizer: RobertaTokenizerFast):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(config.MODEL_NAME)
        self.encoder.resize_token_embeddings(len(tokenizer))
        self.drop = nn.Dropout(0.2)
        hs = self.encoder.config.hidden_size
        self.cls_head = nn.Linear(hs, 2)
        self.va_head = nn.Linear(hs, 2)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.drop(out.last_hidden_state[:, 0, :])
        logits = self.cls_head(cls)
        va = self.va_head(cls)
        return logits, va

def pair_loss(logits, va_pred, y, va_true) -> torch.Tensor:
    loss_cls = nn.CrossEntropyLoss()(logits, y)

    mask = (y == 1)
    if mask.any():
        pred = torch.clamp(va_pred[mask], 1.0, 9.0)
        tgt = va_true[mask]
        loss_va = nn.MSELoss()(pred, tgt)
    else:
        loss_va = torch.zeros((), device=logits.device)

    return loss_cls + loss_va

def train_pair_model(pair_items, tokenizer, wrong_map) -> PairModel:
    ds = PairDataset(pair_items, tokenizer, config.MAX_LEN_PAIR, wrong_map=wrong_map)
    dl = DataLoader(ds, batch_size=config.BATCH_SIZE_PAIR, shuffle=True)

    model = PairModel(tokenizer).to(config.DEVICE)
    opt = AdamW(model.parameters(), lr=config.LR, weight_decay=0.01)

    for epoch in range(config.MAX_EPOCHS_PAIR):
        model.train()
        tot = 0.0
        for batch in tqdm(dl, desc=f"PairModel Epoch {epoch+1}/{config.MAX_EPOCHS_PAIR}"):
            ids = batch["ids"].to(config.DEVICE)
            mask = batch["mask"].to(config.DEVICE)
            y = batch["y"].to(config.DEVICE)
            va = batch["va"].to(config.DEVICE)

            opt.zero_grad()
            logits, va_pred = model(ids, mask)
            loss = pair_loss(logits, va_pred, y, va)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += float(loss.item())

        print(f"   Epoch {epoch+1} | AvgLoss {tot/max(1,len(dl)):.4f}")

    return model

# 9 — Inference (NULL aware)
def infer_dimaste_triplet(
    extractor: ExtractionModel,
    pair_model: PairModel,
    items: List[Dict[str, Any]],
    tokenizer: RobertaTokenizerFast,
) -> Tuple[List[List[Dict[str, Any]]], List[List[Dict[str, Any]]], List[Dict[str, Any]]]:

    cand = extract_candidates(extractor, items, tokenizer)

    preds_per_sent: List[List[Dict[str, Any]]] = []
    golds_per_sent: List[List[Dict[str, Any]]] = []
    rows: List[Dict[str, Any]] = []

    pair_model.eval()
    with torch.no_grad():
        for it in tqdm(items, desc="Inferring DimASTE", leave=False):
            sid = it["id"]
            text = safe_text(it.get("text", ""))

            # Gold includes paired & unpaired as long as VA exists
            golds = [{
                "aspect": normalize_candidate_token(t["aspect"]),
                "opinion": normalize_candidate_token(t["opinion"]),
                "valence": t["valence"],
                "arousal": t["arousal"]
            } for t in it.get("triplets", [])
              if (t.get("valence") is not None and t.get("arousal") is not None)]
            golds_per_sent.append(golds)

            aspects = [normalize_candidate_token(a) for a in cand.get(sid, {}).get("aspects", [])]
            opinions = [normalize_candidate_token(o) for o in cand.get(sid, {}).get("opinions", [])]

            aspects = [a for a in aspects if a and not is_null(a)]
            opinions = [o for o in opinions if o and not is_null(o)]

            sent_preds: List[Dict[str, Any]] = []

            def score_pair(a_: str, o_: str) -> Optional[Dict[str, Any]]:
                if config.USE_NULL_TOKEN:
                    s = f"{NULL_TOKEN} {text} </s></s> {a_} </s></s> {o_}"
                else:
                    s = f"{text} </s></s> {a_} </s></s> {o_}"

                enc = tokenizer(
                    s,
                    padding="max_length",
                    truncation=True,
                    max_length=config.MAX_LEN_PAIR,
                    return_tensors="pt"
                )
                ids = enc["input_ids"].to(config.DEVICE)
                mask = enc["attention_mask"].to(config.DEVICE)

                logits, va = pair_model(ids, mask)
                y = int(torch.argmax(logits, dim=1).item())
                if y == 0:
                    return None

                va = va.squeeze(0).cpu().numpy()
                v = float(np.clip(va[0], 1.0, 9.0))
                ar = float(np.clip(va[1], 1.0, 9.0))

                return {
                    "aspect": a_,
                    "opinion": o_,
                    "valence": float(f"{v:.2f}"),
                    "arousal": float(f"{ar:.2f}")
                }

            if aspects and opinions:
                for a in aspects:
                    for o in opinions:
                        p = score_pair(a, o)
                        if p is not None:
                            sent_preds.append(p)
            elif opinions and not aspects:
                for o in opinions:
                    p = score_pair("NULL", o)
                    if p is not None:
                        sent_preds.append(p)
            elif aspects and not opinions:
                for a in aspects:
                    p = score_pair(a, "NULL")
                    if p is not None:
                        sent_preds.append(p)
            else:
                p = score_pair("NULL", "NULL")
                if p is not None:
                    sent_preds.append(p)

            uniq = {}
            for p in sent_preds:
                k = (norm_text(p["aspect"]), norm_text(p["opinion"]))
                if k not in uniq:
                    uniq[k] = p
            sent_preds = list(uniq.values())

            preds_per_sent.append(sent_preds)

            if sent_preds:
                for p in sent_preds:
                    rows.append({
                        "Sentence_ID": sid,
                        "Text": text,
                        "Predicted_Aspect": p["aspect"],
                        "Predicted_Opinion": p["opinion"],
                        "Predicted_Valence": p["valence"],
                        "Predicted_Arousal": p["arousal"],
                    })
            else:
                rows.append({
                    "Sentence_ID": sid,
                    "Text": text,
                    "Predicted_Aspect": "NULL",
                    "Predicted_Opinion": "NULL",
                    "Predicted_Valence": 5.00,
                    "Predicted_Arousal": 5.00,
                })
    return preds_per_sent, golds_per_sent, rows

# 10 — Metrics
def dimaste_triplet_metrics(preds_per_sent: List[List[Dict[str, Any]]], golds_per_sent: List[List[Dict[str, Any]]]) -> Dict[str, float]:
    TP = FP = FN = 0
    total_dist = 0.0
    matched_va_pairs = []

    for preds, golds in zip(preds_per_sent, golds_per_sent):
        matched = set()
        for p in preds:
            ok = False
            for gi, g in enumerate(golds):
                if gi in matched:
                    continue
                if norm_text(p["aspect"]) == norm_text(g["aspect"]) and norm_text(p["opinion"]) == norm_text(g["opinion"]):
                    TP += 1
                    matched.add(gi)
                    ok = True
                    dist = calculate_normalized_euclidean_distance(p["valence"], p["arousal"], g["valence"], g["arousal"])
                    total_dist += dist
                    matched_va_pairs.append((g["valence"], p["valence"], g["arousal"], p["arousal"]))
                    break
            if not ok:
                FP += 1
        FN += len(golds) - len(matched)

    cTP = TP - total_dist
    cRecall = cTP / (TP + FN) if (TP + FN) > 0 else 0.0
    cPrecision = cTP / (TP + FP) if (TP + FP) > 0 else 0.0
    cF1 = 2 * cRecall * cPrecision / (cRecall + cPrecision) if (cRecall + cPrecision) > 0 else 0.0

    mae_v = mae_a = pv = pa = 0.0
    if len(matched_va_pairs) > 1:
        gv = np.array([x[0] for x in matched_va_pairs], dtype=float)
        pv_ = np.array([x[1] for x in matched_va_pairs], dtype=float)
        ga = np.array([x[2] for x in matched_va_pairs], dtype=float)
        pa_ = np.array([x[3] for x in matched_va_pairs], dtype=float)
        mae_v = float(np.mean(np.abs(gv - pv_)))
        mae_a = float(np.mean(np.abs(ga - pa_)))
        try:
            pv, _ = pearsonr(gv, pv_)
            pa, _ = pearsonr(ga, pa_)
        except Exception:
            pv, pa = 0.0, 0.0

    return {
        "cRecall": float(cRecall),
        "cPrecision": float(cPrecision),
        "cF1": float(cF1),
        "TP_cat": int(TP),
        "FP_cat": int(FP),
        "FN_cat": int(FN),
        "VA_Error_Dist": float(total_dist),
        "MAE_V": float(mae_v),
        "MAE_A": float(mae_a),
        "Pearson_V": float(pv),
        "Pearson_A": float(pa),
    }

# 11 — Main (loop domains)
def run_for_domain(domain: str):
    t0 = time.time()
    print(f"\nRUN DOMAIN: {domain}")

    url = config.get_train_url(domain)
    filename = config.get_local_filename(domain)
    full_data = download_and_load(url, filename)
    if not full_data:
        print("No data loaded.")
        return

    cleaned_data, stats = cleaning_with_stats(full_data)

    trainable_full = filter_trainable_items(cleaned_data)
    trainable_ids = {it["id"] for it in trainable_full}
    non_trainable_items = [it for it in cleaned_data if it["id"] not in trainable_ids]

    print("\nData Separation Summary")
    print(f"Total after cleaning           : {len(cleaned_data)}")
    print(f"Trainable (has VA triplets)    : {len(trainable_full)}")
    print(f"Non-trainable (no VA triplets) : {len(non_trainable_items)}")

    tokenizer = RobertaTokenizerFast.from_pretrained(config.MODEL_NAME)
    if config.USE_NULL_TOKEN:
        if tokenizer.convert_tokens_to_ids(NULL_TOKEN) == tokenizer.unk_token_id:
            tokenizer.add_special_tokens({"additional_special_tokens": [NULL_TOKEN]})
        print("NULL token ready. vocab_size =", len(tokenizer), "| NULL_ID =", tokenizer.convert_tokens_to_ids(NULL_TOKEN))

    K = config.OUTER_K
    kf = KFold(n_splits=K, shuffle=True, random_state=config.SEED)

    all_fold_metrics = []
    all_pred_rows = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(trainable_full), start=1):
        fold_t0 = time.time()

        print(f"\nOUTER FOLD {fold}/{K} ({domain})\n")

        train_data = [trainable_full[i] for i in train_idx]
        test_data = [trainable_full[i] for i in test_idx]

        print(f"TRAIN: {len(train_data)} | TEST: {len(test_data)} | NON-TRAINABLE(extra): {len(non_trainable_items)}")
        print(f"Epochs now: extractor={config.MAX_EPOCHS_EXTRACTION}, pair={config.MAX_EPOCHS_PAIR}, inner_oof_extractor={config.OOF_EPOCHS_EXTRACTION}")
        print(f"Folds now: outer={config.OUTER_K}, inner_oof={config.OOF_K}")

        print("\n[A] Building OOF wrong spans for negatives (INNER KFold)")
        wrong_map = oof_wrong_spans(train_data, tokenizer)

        print("\n[B] Training extractor")
        extractor = train_extractor(train_data, tokenizer, epochs=config.MAX_EPOCHS_EXTRACTION)

        print("\n[C] Training pair model")
        pair_model = train_pair_model(train_data, tokenizer, wrong_map)

        print("\n[D1] Inference on test fold for METRICS")
        preds_t, golds_t, rows_t = infer_dimaste_triplet(extractor, pair_model, test_data, tokenizer)
        m = dimaste_triplet_metrics(preds_t, golds_t)
        all_fold_metrics.append({"Domain": domain, "Fold": fold, **m})

        for r in rows_t:
            r["Domain"] = domain
            r["Fold"] = fold
            r["Split"] = "test_trainable"
        all_pred_rows.extend(rows_t)

        if non_trainable_items:
            print("\n[D2] Extra inference on non-trainable items (no metrics)")
            _, _, rows_nt = infer_dimaste_triplet(extractor, pair_model, non_trainable_items, tokenizer)
            for r in rows_nt:
                r["Domain"] = domain
                r["Fold"] = fold
                r["Split"] = "non_trainable_only"
            all_pred_rows.extend(rows_nt)

        fold_dt = time.time() - fold_t0
        print(f"\nFold {fold} | cF1={m['cF1']:.4f} cRecall={m['cRecall']:.4f} cPrecision={m['cPrecision']:.4f} MAE_V={m['MAE_V']:.4f}")
        print(f"Fold runtime: {format_duration(fold_dt)}")

    pred_path = config.output_predictions_csv(domain)
    metrics_path = config.output_metrics_csv(domain)

    if all_pred_rows:
        dfp = pd.DataFrame(all_pred_rows)
        dfp.to_csv(pred_path, index=False, float_format="%.4f")

    dff = pd.DataFrame(all_fold_metrics)
    metric_cols = [c for c in dff.columns if c not in {"Domain", "Fold"}]

    mean = dff[metric_cols].mean(numeric_only=True).to_dict()
    std = dff[metric_cols].std(numeric_only=True).to_dict()

    dff2 = pd.concat([
        dff,
        pd.DataFrame([{"Domain": domain, "Fold": "MEAN", **mean}]),
        pd.DataFrame([{"Domain": domain, "Fold": "STD", **std}]),
    ], ignore_index=True)

    dff2.to_csv(metrics_path, index=False, float_format="%.4f")

    print("\nFINAL (MEAN ± STD) —", domain)
    print(f"cF1:        {mean.get('cF1',0):.4f} ± {std.get('cF1',0):.4f}")
    print(f"cRecall:    {mean.get('cRecall',0):.4f} ± {std.get('cRecall',0):.4f}")
    print(f"cPrecision: {mean.get('cPrecision',0):.4f} ± {std.get('cPrecision',0):.4f}")
    print(f"MAE_V:      {mean.get('MAE_V',0):.4f} ± {std.get('MAE_V',0):.4f}")
    print(f"MAE_A:      {mean.get('MAE_A',0):.4f} ± {std.get('MAE_A',0):.4f}")

    dt = time.time() - t0

    print(f"DOMAIN {domain} RUNTIME:", format_duration(dt))

def main():
    set_seed(config.SEED)
    t0 = time.time()

    for domain in config.DOMAINS:
        run_for_domain(domain)

    dt = time.time() - t0
    print("TOTAL RUNTIME (ALL DOMAINS):", format_duration(dt))

if __name__ == "__main__":
    main()